In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# SISTEMA DE PREDICAO DE FALHAS EM MAQUINAS INDUSTRIAIS\n",
    "\n",
    "**Autor:** Ivan Manoel dos Santos da Rosa  \n",
    "**LinkedIn:** https://www.linkedin.com/in/ivan-santos-8046a8355/\n",
    "\n",
    "Este projeto implementa uma rede neural para prever falhas em máquinas industriais, utilizando TensorFlow/Keras. O modelo é projetado para auxiliar em manutenção preditiva, reduzindo custos de manutenção não planejada e aumentando a eficiência operacional."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Importação das Bibliotecas"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "# Scikit-learn\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score\n",
    "from sklearn.utils.class_weight import compute_class_weight\n",
    "from sklearn.metrics import ConfusionMatrixDisplay\n",
    "\n",
    "# TensorFlow/Keras\n",
    "import tensorflow as tf\n",
    "from tensorflow.keras.models import Sequential\n",
    "from tensorflow.keras.layers import Dense, Dropout\n",
    "from tensorflow.keras.callbacks import EarlyStopping\n",
    "from tensorflow.keras import regularizers\n",
    "\n",
    "import joblib\n",
    "\n",
    "print(\"Bibliotecas importadas com sucesso!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Carregamento dos Dados"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\" * 70)\n",
    "print(\"Carregando Dataset de Manutenção Preditiva\")\n",
    "print(\"=\" * 70)\n",
    "\n",
    "df = pd.read_csv(\"AI4I.csv\")\n",
    "\n",
    "print(\"\\nPrimeiras linhas do dataset:\")\n",
    "print(df.head())\n",
    "\n",
    "print(f\"\\nDimensões: {df.shape[0]} linhas x {df.shape[1]} colunas\")\n",
    "print(f\"\\nDistribuição da variável alvo (Machine failure):\")\n",
    "print(df[\"Machine failure\"].value_counts(normalize=True).round(3))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Pré-processamento"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Separar features e target\n",
    "X = df.drop([\"UDI\", \"Product ID\", \"Machine failure\"], axis=1)\n",
    "y = df[\"Machine failure\"]\n",
    "\n",
    "# One-hot encoding para variáveis categóricas\n",
    "X = pd.get_dummies(X, columns=[\"Type\"], drop_first=True)\n",
    "\n",
    "print(f\"Features após processamento: {list(X.columns)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Divisão Treino/Teste"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42, stratify=y\n",
    ")\n",
    "\n",
    "print(f\"Treino: {X_train.shape[0]} amostras\")\n",
    "print(f\"Teste: {X_test.shape[0]} amostras\")\n",
    "print(f\"Proporção de falhas no treino: {y_train.mean():.3%}\")\n",
    "print(f\"Proporção de falhas no teste: {y_test.mean():.3%}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Normalização"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "scaler = StandardScaler()\n",
    "X_train_scaled = scaler.fit_transform(X_train)\n",
    "X_test_scaled = scaler.transform(X_test)\n",
    "\n",
    "print(\"Normalização concluída!\")\n",
    "print(f\"Média após normalização: {X_train_scaled.mean():.2e}\")\n",
    "print(f\"Desvio padrão: {X_train_scaled.std():.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Balanceamento de Classes"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class_weights = compute_class_weight(\n",
    "    class_weight=\"balanced\",\n",
    "    classes=np.unique(y_train),\n",
    "    y=y_train\n",
    ")\n",
    "class_weights_dict = dict(enumerate(class_weights))\n",
    "\n",
    "print(\"Pesos das classes para balanceamento:\")\n",
    "for label, weight in class_weights_dict.items():\n",
    "    status = \"Falha\" if label == 1 else \"Normal\"\n",
    "    print(f\"  Classe {label} ({status}): peso = {weight:.2f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Construção da Rede Neural"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model = Sequential([\n",
    "    Dense(32, activation=\"relu\", kernel_regularizer=regularizers.L2(0.001), input_shape=(X_train_scaled.shape[1],)),\n",
    "    Dropout(0.5),\n",
    "    Dense(16, activation=\"relu\", kernel_regularizer=regularizers.L2(0.001)),\n",
    "    Dropout(0.5),\n",
    "    Dense(1, activation=\"sigmoid\")\n",
    "])\n",
    "\n",
    "model.compile(\n",
    "    optimizer=\"adam\",\n",
    "    loss=\"binary_crossentropy\",\n",
    "    metrics=[\"accuracy\"]\n",
    ")\n",
    "\n",
    "print(\"Arquitetura do modelo:\")\n",
    "model.summary()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Treinamento"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "early_stop = EarlyStopping(\n",
    "    monitor=\"val_loss\",\n",
    "    patience=5,\n",
    "    restore_best_weights=True\n",
    ")\n",
    "\n",
    "print(\"Iniciando treinamento...\")\n",
    "history = model.fit(\n",
    "    X_train_scaled, y_train,\n",
    "    epochs=50,\n",
    "    batch_size=16,\n",
    "    validation_split=0.2,\n",
    "    callbacks=[early_stop],\n",
    "    class_weight=class_weights_dict,\n",
    "    verbose=1\n",
    ")\n",
    "\n",
    "print(f\"\\nTreinamento finalizado após {len(history.history['loss'])} épocas\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Salvar Modelo e Scaler"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model.save('modelo_falhas.h5')\n",
    "joblib.dump(scaler, 'scaler.pkl')\n",
    "print(\"Modelo salvo como 'modelo_falhas.h5'\")\n",
    "print(\"Scaler salvo como 'scaler.pkl'\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Avaliação do Modelo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "y_pred_prob = model.predict(X_test_scaled)\n",
    "y_pred = (y_pred_prob > 0.5).astype(int)\n",
    "\n",
    "print(\"\\nRelatório de Classificação (threshold = 0.5):\")\n",
    "print(classification_report(y_test, y_pred))\n",
    "\n",
    "print(\"Matriz de Confusão:\")\n",
    "print(confusion_matrix(y_test, y_pred))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 11. Curva ROC e Otimização do Threshold"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)\n",
    "auc_score = roc_auc_score(y_test, y_pred_prob)\n",
    "\n",
    "print(f\"AUC (Área sob a curva ROC): {auc_score:.3f}\")\n",
    "\n",
    "# Threshold otimizado para priorizar recall\n",
    "y_pred_final = (y_pred_prob > 0.3).astype(int)\n",
    "\n",
    "print(\"\\nRESULTADOS FINAIS (Threshold = 0.3):\")\n",
    "print(\"Matriz de Confusão:\")\n",
    "print(confusion_matrix(y_test, y_pred_final))\n",
    "print(\"\\nRelatório:\")\n",
    "print(classification_report(y_test, y_pred_final))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 12. Visualizações"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(12, 10))\n",
    "\n",
    "# Curva de aprendizado\n",
    "plt.subplot(2, 2, 1)\n",
    "plt.plot(history.history[\"loss\"], label=\"Perda no Treino\", linewidth=2)\n",
    "plt.plot(history.history['val_loss'], label='Perda na Validação', linewidth=2)\n",
    "plt.legend()\n",
    "plt.title('Curva de Aprendizado', fontsize=14, fontweight='bold')\n",
    "plt.xlabel('Época')\n",
    "plt.ylabel('Perda')\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "# Distribuição das probabilidades\n",
    "plt.subplot(2, 2, 2)\n",
    "plt.hist(y_pred_prob[y_test == 0], bins=30, alpha=0.7, label='Sem Falha', color='green')\n",
    "plt.hist(y_pred_prob[y_test == 1], bins=30, alpha=0.7, label='Com Falha', color='red')\n",
    "plt.axvline(x=0.3, color='black', linestyle='--', label='Threshold = 0.3')\n",
    "plt.title('Distribuição das Probabilidades', fontsize=14, fontweight='bold')\n",
    "plt.xlabel('Probabilidade de Falha')\n",
    "plt.ylabel('Frequência')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "# Curva ROC\n",
    "plt.subplot(2, 2, 3)\n",
    "plt.plot(fpr, tpr, label=f\"AUC = {auc_score:.3f}\", linewidth=2, color='blue')\n",
    "plt.plot([0, 1], [0, 1], linestyle=\"--\", color='gray', label='Classificador Aleatório')\n",
    "plt.xlabel(\"Taxa de Falsos Positivos\")\n",
    "plt.ylabel(\"Taxa de Verdadeiros Positivos\")\n",
    "plt.title(\"Curva ROC\", fontsize=14, fontweight='bold')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "# Matriz de Confusão\n",
    "plt.subplot(2, 2, 4)\n",
    "ConfusionMatrixDisplay(\n",
    "    confusion_matrix=confusion_matrix(y_test, y_pred_final),\n",
    "    display_labels=['Operação Normal', 'Falha Detectada']\n",
    ").plot(cmap='Blues', values_format='d', ax=plt.gca())\n",
    "plt.title('Matriz de Confusão Final', fontsize=14, fontweight='bold')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Salvar imagens\n",
    "for i in range(1, 5):\n",
    "    plt.figure(i)\n",
    "    plt.savefig(f'figura_{i}.png', dpi=150, bbox_inches='tight')\n",
    "\n",
    "print(\"Imagens salvas: figura_1.png, figura_2.png, figura_3.png, figura_4.png\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 13. Conclusão"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\" * 70)\n",
    "print(\"CONCLUSÕES\")\n",
    "print(\"=\" * 70)\n",
    "print(f\"\"\"\n",
    "✅ Modelo treinado com sucesso!\n",
    "✅ AUC: {auc_score:.3f}\n",
    "✅ Recall para falhas: 97.1% (threshold = 0.3)\n",
    "✅ Modelo salvo para uso em produção\n",
    "\n",
    "APLICAÇÕES PRÁTICAS:\n",
    "- Redução de custos com manutenção corretiva\n",
    "- Aumento da vida útil dos equipamentos\n",
    "- Planejamento otimizado de manutenção\n",
    "\"\"\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}